# 🏪 SF Registered Business Locations - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

This notebook ingests every **currently open business location** registered with San Francisco's Treasurer & Tax Collector into Elasticsearch: the corner store, the dentist, the coffee roaster, the bike shop. Each record has a name, address, start date, industry code, neighborhood, and (usually) a location.

**What this notebook does:**
1. Fetches open business locations from [DataSF](https://data.sf.gov/d/g8m3-pdis) - no API key required
2. Normalises each record (start dates, industry sector, neighborhood, `geo_point`)
3. Bulk-indexes everything into `sf_business_locations`

---
> **Data source:** Registered Business Locations - San Francisco, dataset `g8m3-pdis`. We keep locations inside San Francisco with no end date (~100k). We leave out owner names and mailing addresses: many small businesses are registered to a person.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Fetch open business locations
~100k rows - it takes a minute. Lower `SAMPLE_SIZE` for a quicker run.

In [ ]:
SAMPLE_SIZE = 150000

def fetch_socrata(dataset_id, limit, page_size=10000, **params):
    """Page through a DataSF dataset with $limit/$offset until `limit` rows or the end."""
    url = f'https://data.sf.gov/resource/{dataset_id}.json'
    rows = []
    while len(rows) < limit:
        page = {**params, '$limit': min(page_size, limit - len(rows)), '$offset': len(rows)}
        r = requests.get(url, params=page, timeout=120)
        r.raise_for_status()
        batch = r.json()
        rows.extend(batch)
        print(f'  fetched {len(rows)} rows...', end='\r', flush=True)
        if len(batch) < page['$limit']:
            break
    return rows

rows = fetch_socrata('g8m3-pdis', SAMPLE_SIZE, page_size=25000, **{
    '$select': 'uniqueid,dba_name,full_business_address,business_zip,dba_start_date,location_start_date,'
               'self_reported_naics_code,lic_code_description,parking_tax,transient_occupancy_tax,'
               'location,business_corridor,supervisor_district,neighborhoods_analysis_boundaries',
    '$where': "location_end_date IS NULL AND dba_end_date IS NULL AND upper(city) = 'SAN FRANCISCO'",
    '$order': 'uniqueid',
})
print(f'\n✅ Fetched {len(rows)} open business locations')

## 3. Create index with explicit mappings

In [ ]:
INDEX = 'sf_business_locations'

mapping = {'mappings': {'properties': {
    'business_id':        {'type': 'keyword'},
    'name':               {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'address':            {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'zip':                {'type': 'keyword'},
    'business_start':     {'type': 'date'},     # when the business (DBA) started
    'location_start':     {'type': 'date'},     # when it opened at this address
    'naics_code':         {'type': 'keyword'},
    'sector':             {'type': 'keyword'},  # human-readable 2-digit NAICS sector
    'license':            {'type': 'text'},     # city license description, when present
    'parking_tax':        {'type': 'boolean'},
    'hotel_tax':          {'type': 'boolean'},
    'business_corridor':  {'type': 'keyword'},
    'neighborhood':       {'type': 'keyword'},
    'supervisor_district':{'type': 'keyword'},
    'location':           {'type': 'geo_point'},
}}}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')
es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

## 4. Enrich and ingest
The raw data has a 4- to 6-digit NAICS industry code; we map its first two digits to a readable **sector** so you can aggregate on it.

In [ ]:
SECTORS = {
    '11': 'Agriculture', '21': 'Mining', '22': 'Utilities', '23': 'Construction',
    '31': 'Manufacturing', '32': 'Manufacturing', '33': 'Manufacturing',
    '42': 'Wholesale Trade', '44': 'Retail Trade', '45': 'Retail Trade',
    '48': 'Transportation & Warehousing', '49': 'Transportation & Warehousing',
    '51': 'Information', '52': 'Finance & Insurance', '53': 'Real Estate',
    '54': 'Professional & Technical Services', '55': 'Management of Companies',
    '56': 'Administrative & Support Services', '61': 'Educational Services',
    '62': 'Health Care & Social Assistance', '71': 'Arts, Entertainment & Recreation',
    '72': 'Accommodation & Food Services', '81': 'Other Services', '92': 'Public Administration',
}

def enrich(r):
    naics = (r.get('self_reported_naics_code') or '').strip()
    coords = (r.get('location') or {}).get('coordinates')
    doc = {
        'business_id':        r.get('uniqueid'),
        'name':               (r.get('dba_name') or '').strip(),
        'address':            (r.get('full_business_address') or '').strip(),
        'zip':                r.get('business_zip'),
        'business_start':     r.get('dba_start_date'),
        'location_start':     r.get('location_start_date'),
        'naics_code':         naics,
        'sector':             SECTORS.get(naics[:2]),
        'license':            r.get('lic_code_description'),
        'parking_tax':        r.get('parking_tax'),
        'hotel_tax':          r.get('transient_occupancy_tax'),
        'business_corridor':  r.get('business_corridor'),
        'neighborhood':       r.get('neighborhoods_analysis_boundaries'),
        'supervisor_district':r.get('supervisor_district'),
        'location':           {'lat': coords[1], 'lon': coords[0]} if coords else None,
    }
    return {k: v for k, v in doc.items() if v not in (None, '')}

docs = [enrich(r) for r in rows]
success, errors = helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} documents into {INDEX} ({len(errors)} errors)')

## 5. Verify

In [ ]:
resp = es.search(index=INDEX, body={'size': 0, 'track_total_hits': True, 'aggs': {
    'by_sector':   {'terms': {'field': 'sector', 'size': 10}},
    'by_corridor': {'terms': {'field': 'business_corridor', 'size': 10}},
}})
print(f"Total open locations: {resp['hits']['total']['value']}\n")
print('Top sectors:')
for b in resp['aggregations']['by_sector']['buckets']:
    print(f"  {b['key']:36s} {b['doc_count']}")
print('\nBusiest commercial corridors:')
for b in resp['aggregations']['by_corridor']['buckets']:
    print(f"  {b['key']:36s} {b['doc_count']}")

In [ ]:
# The longest-running restaurants, cafes, and bars still open (NAICS 722 = food services & drinking places).
# The registry's dates start in late 1968, so the oldest places all show 1968.
resp = es.search(index=INDEX, body={'size': 8,
    'query': {'bool': {'filter': [{'prefix': {'naics_code': '722'}},
                                  {'exists': {'field': 'location'}}]}},
    'sort': [{'business_start': 'asc'}],
    '_source': ['name', 'address', 'business_start', 'neighborhood'],
})
print('Oldest food & drink businesses still open:')
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {s['business_start'][:4]}  {s['name'][:40]:40s} {s.get('neighborhood', '')}")

## 6. Going further with Mistral

- **Find a local alternative:** semantic search over business names + sectors with Mistral embeddings ("somewhere to fix my bike near Dolores Park"), then a Mistral chat model writes the recommendation. See [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md).
- **Small-business helper:** a Mistral agent that drafts a shop's announcement, translates it into Spanish, Chinese, Tagalog or Vietnamese, and uses this index to suggest which nearby businesses to partner with.
- **Neighborhood change:** aggregate `location_start` by year and corridor, and have Mistral narrate how Valencia Street or Clement Street has changed.
- **Combine:** join with [sf_health_inspections](sf_health_inspections.ipynb) or [sf_311_cases](sf_311_cases.ipynb) by location.

See [mistral_guide.md](mistral_guide.md) for calling Mistral directly.

---
## ✅ Index loaded
`sf_business_locations` is ready. Try "coffee shops on Valencia that opened before 2000" or "how many restaurants are within 500 m of the venue?"

**To refresh the data**, re-run cells 2-4. The index is dropped and recreated each time, so there are no duplicates.